In [ ]:
# file modified from original Apache-2.0 licensed code from https://github.com/Understanding-Visual-Datasets/VisDiff
# see LICENSE and NOTICE files in the root directory for details


In [ ]:
# code adapted from https://github.com/Understanding-Visual-Datasets/VisDiff/issues/9#issuecomment-2558625174

In [ ]:
import pandas as pd
import wandb
import numpy as np
from datetime import datetime
from typing import Dict, List, Tuple, Union

api = wandb.Api()

In [ ]:

def get_logs(wandb_entity_name:str, wandb_project_name:str, group_name:str|None = None, created_earliest: datetime|None = None, created_latest:datetime|None = None, filter_state:bool = False):
    filters = {
        "$and": []
    }
    if group_name is not None:
        filters["$and"].append({"group": group_name})
    if created_earliest is not None:
        filters["$and"].append({"createdAt": {"$gte": created_earliest.isoformat()}})
    if created_latest is not None:
        filters["$and"].append({"createdAt": {"$lte": created_latest.isoformat()}})
    if filter_state:
        filters["$and"].append({"state": "finished"})
    if len(filters["$and"]) == 0:
        filters = {}

    runs = api.runs(path=f"{wandb_entity_name}/{wandb_project_name}", filters=filters)

    summary_list, config_list, name_list = [], [], []
    for run in runs:
        # .summary contains the output keys/values for metrics like accuracy.
        #  We call ._json_dict to omit large files
        summary_list.append(run.summary._json_dict)

        # .config contains the hyperparameters.
        #  We remove special values that start with _.
        config_list.append(
            {k: v for k, v in run.config.items() if not k.startswith("_")}
        )

        # .name is the human-readable name of the run.
        name_list.append(run.name)

    runs_df = pd.DataFrame(
        {"summary": summary_list, "config": config_list, "name": name_list}
    )
    csv_name = f"{wandb_project_name}{'_' + group_name.replace(' ', '_') if group_name else ''}{'_' + created_earliest.strftime('%Y%m%d%H%M%S') if created_earliest else ''}{'_' + created_latest.strftime('%Y%m%d%H%M%S') if created_latest else ''}.csv"
    runs_df.to_csv(csv_name)
    return runs_df.to_dict("records")


def get_metrics(
    results: List[Dict], keys: List[str] = ["acc@1", "acc@5", "acc@N"]
) -> Dict:
    def compute_metrics(results_or_subresults: List[Dict], keys: List[str]) -> Dict:
        metrics = {}
        for result in results_or_subresults:
            try:
                for key in keys:
                    if key not in metrics:
                        metrics[key] = []
                    metrics[key].append(result["summary"][key])
            except Exception as e:
                continue
        for key in keys:
            metrics[key] = {
                "mean": np.mean(metrics[key]),
                "std": np.std(metrics[key]),
                "n": len(metrics[key]),
            }
        return metrics
    if any((result["config"].get("same_diff_group_name") is not None for result in results)):
        assert all((result["config"].get("same_diff_group_name") is not None for result in results)), "Either all or none of the results must have 'same_diff_group_name' in config."
        same_diff_groups = dict()
        for result in results:
            group_name = result["config"]["same_diff_group_name"]
            if group_name not in same_diff_groups:
                same_diff_groups[group_name] = []
            same_diff_groups[group_name].append(result)
        #metrics = {group_name: compute_metrics(group_results, keys) for group_name, group_results in same_diff_groups.items()}
        sub_metrics = [compute_metrics(group_results, keys) for group_name, group_results in same_diff_groups.items()]
        metrics = dict()
        for key in keys:
            metrics[key] = {
                "mean": np.mean([sub_metric[key]["mean"] for sub_metric in sub_metrics]),
                "std": np.std([sub_metric[key]["mean"] for sub_metric in sub_metrics]),
                "n": len(sub_metrics),
            }
    else:
        metrics = compute_metrics(results, keys)
    return metrics


In [ ]:
def show_results_scraped_sets(
    wandb_entity_name:str,
    wandb_project_name:str,
    group_name:str|None = None,
    created_earliest: datetime|None = None,
    created_latest:datetime|None = None,
    filter_state:bool = True):
    results = get_logs(
        wandb_entity_name,
        wandb_project_name,
        group_name=group_name,
        created_earliest=created_earliest,
        created_latest=created_latest,
        filter_state=filter_state,
    )
    easy_results = [result for result in results if "easy" in result["config"]["config"]]
    medium_results = [result for result in results if "medium" in result["config"]["config"]]
    hard_results = [result for result in results if "hard" in result["config"]["config"]]
    expected_num_results = 60 if wandb_project_name == "AD-Diff_Bench" else 50
    if any(len(res) != expected_num_results for res in [easy_results, medium_results, hard_results]):
        print(f"Warning: Not all sets have {expected_num_results} results. Check the data. Runs may be missing or results of different sweeps may be mixed. Numbers of results:")
        print(f"Easy: {len(easy_results)}, Medium: {len(medium_results)}, Hard: {len(hard_results)}")

    print("Easy set results:")
    print(get_metrics(easy_results))
    print("Medium set results:")
    print(get_metrics(medium_results))
    print("Hard set results:")
    print(get_metrics(hard_results))
    print("Overall results:")
    print(get_metrics(results))

def show_results_ad_dataset_sets(
    wandb_entity_name:str,
    wandb_project_name:str,
    group_name:str|None,
    filter_state:bool = True):
    results = get_logs(
        wandb_entity_name,
        wandb_project_name,
        group_name=group_name,
        filter_state=filter_state,
    )
    metrics = get_metrics(results)
    print(f"Results for group {group_name}:")
    print(metrics)